# GSEA Proteomics Agent — AWS Bedrock AgentCore

Adaptation of `06_gsea_proteomics.ipynb` using **AWS Bedrock AgentCore** as the LLM backend.

**Changes from original:**
- LLM: OpenAI GPT-4o → `boto3` Bedrock Converse API (Claude 3.5 Sonnet v2)
- Tooling: LangGraph `ToolNode` with `@tool` decorator
- Deployment: Wrapped as an **AgentCore** entrypoint for production serving

**Prerequisites:**
```bash
pip install langchain-aws boto3 gseapy cptac langgraph
pip install amazon-bedrock-agentcore  # optional: for AgentCore runtime
```

Set AWS credentials via `~/.aws/credentials` or environment variables:
```bash
export AWS_DEFAULT_REGION=us-east-1
export AWS_ACCESS_KEY_ID=...
export AWS_SECRET_ACCESS_KEY=...
```

## 1. Install / imports

In [ ]:
# Uncomment to install missing packages
# !pip install langchain-aws boto3 gseapy cptac langgraph
# !pip install amazon-bedrock-agentcore  # AgentCore runtime SDK

In [ ]:
import os, json, uuid
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import gseapy as gp

import boto3
from botocore.exceptions import ClientError

from typing import Annotated, List, Union
from typing_extensions import TypedDict
from pydantic import BaseModel, ValidationError

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

print("Imports OK")

## 2. AWS Bedrock — configure client & model

In [ ]:
# ── AWS configuration ──────────────────────────────────────────────────────────
AWS_REGION   = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# Claude 3.5 Sonnet v2 — supports tool use natively on Bedrock
BEDROCK_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

print(f"Bedrock client ready  |  region={AWS_REGION}  |  model={BEDROCK_MODEL_ID}")

## 3. Custom Bedrock LLM — LangChain-compatible wrapper

Wraps `bedrock-runtime` **Converse API** so it can be used as a drop-in
replacement for `ChatOpenAI` inside LangGraph.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
import copy


def _lc_to_bedrock_messages(messages: list) -> tuple[str, list]:
    """Convert LangChain message list → (system_prompt, bedrock_messages)."""
    system_prompt = ""
    bedrock_msgs = []
    for msg in messages:
        if isinstance(msg, SystemMessage):
            system_prompt = msg.content
        elif isinstance(msg, HumanMessage):
            bedrock_msgs.append({"role": "user",      "content": [{"text": msg.content}]})
        elif isinstance(msg, AIMessage):
            content_blocks = []
            if msg.content:
                content_blocks.append({"text": msg.content if isinstance(msg.content, str)
                                        else " ".join(b.get("text", "") for b in msg.content
                                                      if isinstance(b, dict))})
            # attach tool-use blocks
            for tc in (msg.tool_calls or []):
                content_blocks.append({
                    "toolUse": {
                        "toolUseId": tc["id"],
                        "name":      tc["name"],
                        "input":     tc["args"],
                    }
                })
            if content_blocks:
                bedrock_msgs.append({"role": "assistant", "content": content_blocks})
        elif isinstance(msg, ToolMessage):
            bedrock_msgs.append({
                "role": "user",
                "content": [{
                    "toolResult": {
                        "toolUseId": msg.tool_call_id,
                        "content":   [{"text": str(msg.content)}],
                    }
                }]
            })
    return system_prompt, bedrock_msgs


def _lc_tools_to_bedrock(tools: list) -> list:
    """Convert LangChain tool list → Bedrock toolConfig spec."""
    bedrock_tools = []
    for t in tools:
        schema = t.args_schema.schema() if t.args_schema else {"type": "object", "properties": {}}
        bedrock_tools.append({
            "toolSpec": {
                "name":        t.name,
                "description": t.description,
                "inputSchema": {"json": schema},
            }
        })
    return bedrock_tools


class ChatBedrock(BaseChatModel):
    """Minimal LangChain-compatible wrapper around Bedrock Converse API."""

    model_id:   str = BEDROCK_MODEL_ID
    region:     str = AWS_REGION
    temperature: float = 0.0
    max_tokens:  int   = 4096
    _bound_tools: list = []

    @property
    def _llm_type(self) -> str:
        return "bedrock-converse"

    def bind_tools(self, tools, **kwargs):
        clone = copy.copy(self)
        clone._bound_tools = list(tools)
        return clone

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        system_prompt, bedrock_msgs = _lc_to_bedrock_messages(messages)

        call_kwargs: dict = {
            "modelId":       self.model_id,
            "messages":      bedrock_msgs,
            "inferenceConfig": {
                "maxTokens":   self.max_tokens,
                "temperature": self.temperature,
            },
        }
        if system_prompt:
            call_kwargs["system"] = [{"text": system_prompt}]
        if self._bound_tools:
            call_kwargs["toolConfig"] = {"tools": _lc_tools_to_bedrock(self._bound_tools)}

        response  = bedrock_client.converse(**call_kwargs)
        output    = response["output"]["message"]
        stop_reason = response.get("stopReason", "")

        # Build AIMessage with optional tool_calls
        text_parts  = []
        tool_calls  = []
        for block in output.get("content", []):
            if "text" in block:
                text_parts.append(block["text"])
            elif "toolUse" in block:
                tu = block["toolUse"]
                tool_calls.append({
                    "id":   tu["toolUseId"],
                    "name": tu["name"],
                    "args": tu["input"],
                    "type": "tool_call",
                })

        ai_msg = AIMessage(
            content    = " ".join(text_parts),
            tool_calls = tool_calls,
        )
        return ChatResult(generations=[ChatGeneration(message=ai_msg)])


llm = ChatBedrock(model_id=BEDROCK_MODEL_ID, temperature=0)
print("ChatBedrock wrapper ready")

## 4. Load CPTAC UCEC proteomics data

In [ ]:
import cptac
en = cptac.Ucec()

tumorProt = en.join_metadata_to_omics(
    metadata_name   = "clinical",
    metadata_source = "mssm",
    metadata_cols   = "type_of_analyzed_samples",
    omics_name      = "proteomics",
    omics_source    = "umich",
)

# Remove duplicated columns
tumorProt = tumorProt.loc[:, ~tumorProt.columns.duplicated()]

meta_col = "type_of_analyzed_samples_mssm_clinical"
tumor  = tumorProt[tumorProt[meta_col] == "Tumor"]
normal = tumorProt[tumorProt[meta_col] != "Tumor"]

print(f"Tumor samples: {len(tumor)}  |  Normal samples: {len(normal)}")

## 5. Differential protein expression — Welch's t-test

In [ ]:
tumor_genes  = []
normal_genes = []
genes        = tumorProt.columns[1:]          # skip the meta column
threshold    = 0.05 / len(genes)              # Bonferroni correction

for gene in genes:
    tg = tumor[gene]
    ng = normal[gene]
    if len(tg.shape) > 1:  tg = tg.mean(axis=1)
    if len(ng.shape) > 1:  ng = ng.mean(axis=1)

    pvalue = stats.ttest_ind(tg, ng, equal_var=False, nan_policy="omit").pvalue
    if pvalue < threshold:
        gene_name = gene.split("_")[0]
        if tg.mean() > ng.mean():
            tumor_genes.append(gene_name)
        else:
            normal_genes.append(gene_name)

print(f"Tumor-elevated proteins : {len(tumor_genes)}")
print(f"Normal-elevated proteins: {len(normal_genes)}")

## 6. Baseline enrichment (no agent) — sanity check

In [ ]:
from gseapy.plot import barplot

tumor_enr  = gp.enrichr(gene_list=tumor_genes,  gene_sets="KEGG_2016", outdir="test/enrichr_kegg_tumor")
normal_enr = gp.enrichr(gene_list=normal_genes, gene_sets="KEGG_2016", outdir="test/enrichr_kegg_normal")

print("Top tumor pathways:")
print(tumor_enr.res2d[["Term", "Adjusted P-value", "Genes"]].head())

barplot(tumor_enr.res2d,  title="Proteomics Tumor — KEGG 2016")
plt.show()
barplot(normal_enr.res2d, title="Proteomics Normal — KEGG 2016")
plt.show()

## 7. Enrichr tool — LangGraph `@tool` decorator

In [ ]:
class EnrichrInput(BaseModel):
    inputgenelist: List[str]
    gene_sets:     str = "KEGG_2016"
    top_n:         int = 5


@tool(args_schema=EnrichrInput)
def enrichr_tool(inputgenelist: List[str], gene_sets: str = "KEGG_2016", top_n: int = 5) -> dict:
    """
    Perform gene-set enrichment analysis (Enrichr) on a list of gene symbols.
    Returns the top enriched pathways from the specified gene-set library.

    Args:
        inputgenelist: list of HGNC gene symbols (e.g. ['KRAS', 'TP53', 'EGFR'])
        gene_sets:     Enrichr library name (default: 'KEGG_2016')
        top_n:         number of top results to return (default: 5)

    Returns:
        dict with key 'enrichment_results' containing top_n enriched terms.
    """
    try:
        enr = gp.enrichr(
            gene_list  = inputgenelist,
            gene_sets  = gene_sets,
            outdir     = f"test/enrichr_{gene_sets.lower()}",
        )
        results = enr.res2d.head(top_n).to_dict(orient="records")
        return {"enrichment_results": results}
    except Exception as e:
        return {"error": str(e)}


tools = [enrichr_tool]
llm_with_tools = llm.bind_tools(tools)

# Quick smoke-test
test = enrichr_tool.invoke({"inputgenelist": ["KRAS", "TP53", "EGFR", "PTEN"]})
print("Tool smoke-test OK — top term:", test["enrichment_results"][0]["Term"])

## 8. LangGraph agent — Bedrock backbone

In [ ]:
SYSTEM_PROMPT = (
    "You are a bioinformatics assistant specializing in gene-set enrichment analysis (GSEA). "
    "When the user provides a gene list, call the enrichr_tool to retrieve enriched pathways, "
    "then interpret and summarize the biological significance of the top results."
)

memory = MemorySaver()


def call_model(state: MessagesState) -> dict:
    """Agent node: prepend system prompt and invoke the Bedrock LLM."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


tool_node = ToolNode(tools)

workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

app = workflow.compile(checkpointer=memory)

try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    pass

print("LangGraph agent compiled with Bedrock backbone")

## 9. Run the agent — example query

In [ ]:
target_genes = [
    "HBG1", "GATA2", "ANKRD22", "LHX4", "PSMD9", "APH1A", "TRAPPC4",
    "KRAS", "TP53", "PTCH1", "DNMT3A", "PTPRS", "JAK2", "TNFAIP3",
    "BRCA1", "ASXL1", "EGFR", "PTEN",
]

thread_id = str(uuid.uuid4())
config    = {"configurable": {"thread_id": thread_id}}

user_query = (
    f"Please run enrichment analysis on this gene list and summarize the key pathways: "
    f"{target_genes}"
)

response = app.invoke(
    {"messages": [HumanMessage(content=user_query)]},
    config = config,
)

print("=" * 60)
for msg in response["messages"]:
    role = msg.__class__.__name__.upper().replace("MESSAGE", "")
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}]\n{msg.content}\n")

## 10. Run agent on tumor-elevated proteomics genes

In [ ]:
thread_id2 = str(uuid.uuid4())
config2    = {"configurable": {"thread_id": thread_id2}}

prot_query = (
    f"I have {len(tumor_genes)} proteins significantly elevated in UCEC tumors vs normal tissue. "
    f"Run KEGG enrichment on the first 200 and tell me the top biological themes. "
    f"Gene list: {tumor_genes[:200]}"
)

prot_response = app.invoke(
    {"messages": [HumanMessage(content=prot_query)]},
    config = config2,
)

print("=" * 60)
for msg in prot_response["messages"]:
    role = msg.__class__.__name__.upper().replace("MESSAGE", "")
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}]\n{msg.content}\n")

## 11. Interactive streaming loop

In [ ]:
def stream_agent(user_input: str, thread_id: str = None) -> None:
    """Stream the agent response token by token."""
    tid    = thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": tid}}

    for event in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config = config,
    ):
        for node, state in event.items():
            if node == "agent":
                last = state["messages"][-1]
                if isinstance(last, AIMessage) and last.content:
                    print(f"[AGENT] {last.content}")
            elif node == "tools":
                for m in state["messages"]:
                    if isinstance(m, ToolMessage):
                        snippet = m.content[:200] + "..." if len(m.content) > 200 else m.content
                        print(f"[TOOL:{m.name}] {snippet}")


# Example — comment out in batch mode
stream_agent("What pathways are enriched in: KRAS, EGFR, PTEN, TP53, MYC, BRCA1, CDK4?")

## 12. AWS Bedrock AgentCore — production deployment

AgentCore wraps the LangGraph agent as a managed runtime endpoint.
This cell shows the deployment pattern; it requires the
`amazon-bedrock-agentcore` SDK and AWS IAM permissions for `bedrock:InvokeAgent`.

```bash
pip install amazon-bedrock-agentcore
```

In [ ]:
# ── AgentCore entrypoint ──────────────────────────────────────────────────────
# The AgentCore SDK expects an `app` object with an `@app.entrypoint` handler.
# Below shows the pattern; uncomment and run when the SDK is available.

try:
    from bedrock_agentcore import BedrockAgentCoreApp
    agentcore_app = BedrockAgentCoreApp()

    @agentcore_app.entrypoint
    def gsea_agent_handler(payload: dict) -> dict:
        """
        AgentCore entrypoint.
        Expected payload: {"query": "<user question>", "thread_id": "<optional>"}
        """
        query     = payload.get("query", "")
        thread_id = payload.get("thread_id", str(uuid.uuid4()))
        config    = {"configurable": {"thread_id": thread_id}}

        result = app.invoke(
            {"messages": [HumanMessage(content=query)]},
            config = config,
        )
        final_msg = result["messages"][-1]
        return {
            "answer":    final_msg.content,
            "thread_id": thread_id,
        }

    print("AgentCore app registered — run `agentcore_app.run()` to start the server.")
    # agentcore_app.run()   # start local AgentCore runtime

except ImportError:
    print(
        "amazon-bedrock-agentcore SDK not installed.\n"
        "Install with: pip install amazon-bedrock-agentcore\n"
        "The LangGraph agent above still works directly via Bedrock Converse API."
    )

## 13. Invoke AgentCore via boto3 (after deployment)

Once deployed to AWS, you can invoke the agent via the `bedrock-agentcore` boto3 client.

In [ ]:
def invoke_agentcore(
    agent_id:   str,
    alias_id:   str,
    query:      str,
    session_id: str = None,
) -> str:
    """
    Call a deployed AgentCore agent via the Bedrock Agents Runtime API.

    Parameters
    ----------
    agent_id   : AgentCore agent ID (from AWS console)
    alias_id   : Agent alias ID
    query      : Natural-language user query
    session_id : Optional conversation session ID
    """
    agents_client = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
    sid  = session_id or str(uuid.uuid4())

    response = agents_client.invoke_agent(
        agentId        = agent_id,
        agentAliasId   = alias_id,
        sessionId      = sid,
        inputText      = query,
    )

    # Collect streaming chunks
    full_response = ""
    for event in response["completion"]:
        if "chunk" in event:
            full_response += event["chunk"]["bytes"].decode("utf-8")

    return full_response


# Example (replace with your deployed agent IDs):
# result = invoke_agentcore(
#     agent_id   = "XXXXXXXXXX",
#     alias_id   = "YYYYYYYYYY",
#     query      = "Run enrichment on KRAS, TP53, EGFR, PTEN and summarize pathways.",
# )
# print(result)

print("invoke_agentcore() helper defined — fill in agent_id and alias_id after deployment.")